# Agent 2: Matcher & Explainer (starter)

Compares one synthetic patient against a trial's structured rule list and
returns Met / Not Met / Unknown with a one-line reason per rule, matching
the example table in the proposal.

**Hybrid approach:**
- `numeric` / `categorical` / `categorical_contains` rules -> plain Python (no AI, always consistent)
- `text_interpretation` rules -> LLM call (falls back to "Unknown" if no API key is set)

To enable the LLM path: `pip install anthropic python-dotenv`, get an API key from
console.anthropic.com, then create a `.env` file (add it to `.gitignore` first!)
containing `ANTHROPIC_API_KEY=your-key-here`.


In [1]:
import json
import os
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # dotenv not installed -- fine, just won't auto-load a .env file

API_KEY = os.environ.get("ANTHROPIC_API_KEY")

try:
    import anthropic
    _client = anthropic.Anthropic(api_key=API_KEY) if API_KEY else None
except ImportError:
    _client = None


In [2]:
def get_nested(record: dict, dotted_field: str):
    """Support fields like 'biomarkers.HER2_status' against a nested dict."""
    value = record
    for part in dotted_field.split("."):
        if isinstance(value, dict) and part in value:
            value = value[part]
        else:
            return None
    return value


In [3]:
def check_numeric(patient, rule):
    field_value = get_nested(patient, rule["field"])
    if field_value is None:
        return "Unknown", f"Patient record has no value for '{rule['field']}'"

    op = rule["operator"]
    target = rule["value"]
    ok = {
        ">=": field_value >= target,
        "<=": field_value <= target,
        ">": field_value > target,
        "<": field_value < target,
        "==": field_value == target,
    }[op]

    if ok:
        return "Met", f"{rule['field']} = {field_value}, satisfies {op} {target}"
    return "Not Met", f"{rule['field']} = {field_value}, does not satisfy {op} {target}"


In [4]:
def check_categorical(patient, rule):
    field_value = get_nested(patient, rule["field"])
    if field_value is None:
        return "Unknown", f"Patient record has no value for '{rule['field']}'"

    if field_value == rule["value"]:
        return "Met", f"{rule['field']} = '{field_value}', matches required '{rule['value']}'"
    return "Not Met", f"{rule['field']} = '{field_value}', does not match required '{rule['value']}'"


def check_categorical_contains(patient, rule):
    field_value = get_nested(patient, rule["field"])
    if field_value is None:
        return "Unknown", f"Patient record has no value for '{rule['field']}'"

    items = field_value if isinstance(field_value, list) else [field_value]
    contains = any(rule["value"].lower() in str(item).lower() for item in items)

    if rule["operator"] == "not_contains":
        if contains:
            return "Not Met", f"'{rule['value']}' found in {rule['field']}: {items}"
        return "Met", f"'{rule['value']}' not found in {rule['field']}: {items}"
    else:  # "contains"
        if contains:
            return "Met", f"'{rule['value']}' found in {rule['field']}: {items}"
        return "Not Met", f"'{rule['value']}' not found in {rule['field']}: {items}"


In [5]:
def check_text_interpretation(patient, rule):
    """The fuzzy path -- ask the LLM to interpret free-text-style rules
    against the patient's data. Falls back to Unknown if no API key is set."""
    if _client is None:
        return "Unknown", "LLM not configured (no ANTHROPIC_API_KEY set) -- needs manual review"

    prompt = f"""You are checking one clinical trial eligibility rule against one patient record.

Rule: {rule['raw_text']}

Patient record (JSON):
{json.dumps(patient, indent=2)}

Decide if the rule is Met, Not Met, or Unknown (use Unknown only if the patient
record genuinely lacks the information needed to decide -- do not guess).
Respond with EXACTLY two lines, nothing else:
RESULT: <Met|Not Met|Unknown>
REASON: <one short sentence>"""

    try:
        response = _client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=150,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        result_line = next(l for l in text.splitlines() if l.startswith("RESULT:"))
        reason_line = next(l for l in text.splitlines() if l.startswith("REASON:"))
        result = result_line.split("RESULT:", 1)[1].strip()
        reason = reason_line.split("REASON:", 1)[1].strip()
        return result, reason
    except Exception as e:
        return "Unknown", f"LLM call failed ({e}) -- needs manual review"


In [6]:
CHECKERS = {
    "numeric": check_numeric,
    "categorical": check_categorical,
    "categorical_contains": check_categorical_contains,
    "text_interpretation": check_text_interpretation,
}


def match_patient_to_trial(patient: dict, trial: dict):
    results = []
    for rule in trial["rules"]:
        checker = CHECKERS[rule["check_type"]]
        result, reason = checker(patient, rule)
        results.append({
            "rule_id": rule["rule_id"],
            "description": rule["description"],
            "result": result,
            "reason": reason,
        })
    return results


def summarize(results):
    met = sum(1 for r in results if r["result"] == "Met")
    not_met = sum(1 for r in results if r["result"] == "Not Met")
    unknown = sum(1 for r in results if r["result"] == "Unknown")
    return f"{met} rules met, {not_met} not met, {unknown} unknown"


## Load data and run

Loads the sample trial criteria and your synthetic patients, then runs the
first 3 patients against the sample trial as a demo.


In [7]:
with open("sample_trial_criteria.json") as f:
    trial = json.load(f)

with open("synthetic_patients_simple.json") as f:
    patients = json.load(f)

#for patient in patients[:3]:
   # print(f"\n=== {patient['patient_id']} vs. {trial['trial_name']} ===")
   # results = match_patient_to_trial(patient, trial)
   # for r in results:
       # print(f"  [{r['result']:8}] {r['description']}  -- {r['reason']}")
    #print(f"  Summary: {summarize(results)}")
rows = []
for patient in patients[:3]:
    results = match_patient_to_trial(patient, trial)
    for r in results:
        rows.append({
            "patient_id": patient["patient_id"],
            "rule": r["description"],
            "result": r["result"],
            "reason": r["reason"],
        })

df = pd.DataFrame(rows)

def color_result(val):
    color = {"Met": "#c6efce", "Not Met": "#ffc7ce", "Unknown": "#ffeb9c"}.get(val, "")
    return f"background-color: {color}"
df.style.map(color_result, subset=["result"])

,patient_id,rule,result,reason
0,SYN-001,Participants must be at least 18 years old,Met,"age = 68, satisfies >= 18"
1,SYN-001,Diagnosis must be HER2-positive breast cancer,Not Met,"biomarkers.HER2_status = 'Negative', does not match required 'Positive'"
2,SYN-001,No chemotherapy within the last 4 weeks prior to enrollment,Met,"The patient's prior treatments list shows ""none"", indicating no chemotherapy has been received."
3,SYN-001,ECOG performance status must be 0 or 1,Not Met,"ecog_performance_status = 3, does not satisfy <= 1"
4,SYN-001,No previous immunotherapy,Met,'immunotherapy' not found in prior_treatments: ['none']
5,SYN-002,Participants must be at least 18 years old,Met,"age = 42, satisfies >= 18"
6,SYN-002,Diagnosis must be HER2-positive breast cancer,Not Met,"biomarkers.HER2_status = 'Negative', does not match required 'Positive'"
7,SYN-002,No chemotherapy within the last 4 weeks prior to enrollment,Met,"The patient's prior treatments list contains ""none"", indicating no chemotherapy has been received, which satisfies the requirement of not receiving chemotherapy within 4 weeks prior to enrollment."
8,SYN-002,ECOG performance status must be 0 or 1,Met,"ecog_performance_status = 0, satisfies <= 1"
9,SYN-002,No previous immunotherapy,Met,'immunotherapy' not found in prior_treatments: ['none']


In [8]:
overall_met = overall_not_met = overall_unknown = 0
rows = []
for patient in patients:
    results = match_patient_to_trial(patient, trial)
    m = sum(1 for r in results if r["result"] == "Met")
    nm = sum(1 for r in results if r["result"] == "Not Met")
    u = sum(1 for r in results if r["result"] == "Unknown")
    overall_met += m
    overall_not_met += nm
    overall_unknown += u
    rows.append({
        "patient_id": patient["patient_id"],
        "met": m, "not_met": nm, "unknown": u,
        "overall": "Likely Eligible" if nm == 0 else "Excluded",
    })

summary_df = pd.DataFrame(rows)
print(f"Met: {overall_met}, Not Met: {overall_not_met}, Unknown: {overall_unknown}")
summary_df

Met: 58, Not Met: 29, Unknown: 13


,patient_id,met,not_met,unknown,overall
0,SYN-001,3,2,0,Excluded
1,SYN-002,4,1,0,Excluded
2,SYN-003,4,1,0,Excluded
3,SYN-004,3,1,1,Excluded
4,SYN-005,3,1,1,Excluded
5,SYN-006,4,1,0,Excluded
6,SYN-007,1,3,1,Excluded
7,SYN-008,4,1,0,Excluded
8,SYN-009,3,2,0,Excluded
9,SYN-010,2,2,1,Excluded


In [2]:
import os

def find_project_root(marker="data/trials"):
    path = os.getcwd()
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    return None

root = find_project_root()
if root is None:
    print("Could not auto-locate 'data/trials' -- check your folder structure manually")
else:
    trial_dir = os.path.join(root, "data", "trials")
    print(f"Found trials folder at: {trial_dir}")

Found trials folder at: /Users/sumiya/Desktop/Capstone project /clinical repo/capstone_project/data/trials


In [5]:
import json
import glob

VALID_CHECK_TYPES = {"numeric", "categorical", "categorical_contains", "text_interpretation"}

trial_files = sorted(glob.glob(os.path.join(trial_dir, "*.json")))
print(f"Found {len(trial_files)} trial files\n")

trials = []
for path in trial_files:
    try:
        with open(path) as f:
            trial = json.load(f, strict=False)  # tolerates raw control chars/newlines in strings
    except json.JSONDecodeError as e:
        print(f"❌ {os.path.basename(path)} has invalid JSON: {e} -- skipped")
        continue

    missing_keys = [k for k in ("trial_id", "trial_name", "rules") if k not in trial]
    if missing_keys:
        print(f"⚠️  {os.path.basename(path)} is missing keys: {missing_keys} -- skipped")
        continue

    bad_rules = [r.get("check_type") for r in trial["rules"] if r.get("check_type") not in VALID_CHECK_TYPES]
    if bad_rules:
        print(f"⚠️  {trial['trial_id']} has unrecognized check_type values: {set(bad_rules)}")

    trials.append(trial)
    print(f"✅ {trial['trial_id']} -- {trial['trial_name']} ({len(trial['rules'])} rules)")

print(f"\n{len(trials)} of {len(trial_files)} trials loaded successfully")

Found 10 trial files

⚠️  NCT05950945.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT06120283.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT06625775.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT06756932.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT06757634.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT06982521.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT07062965.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT07358377.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT07368998.json is missing keys: ['trial_name', 'rules'] -- skipped
⚠️  NCT07369505.json is missing keys: ['trial_name', 'rules'] -- skipped

0 of 10 trials loaded successfully
